In [35]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [26]:
load_dotenv() 
llm = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0.2)

In [27]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [28]:
def chat_node(state: ChatState):

    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages': [response]}

In [37]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chat_bot = graph.compile(checkpointer=checkpointer)

In [34]:
from langchain_core.messages import SystemMessage, HumanMessage

initial_state = {
    "messages": [
        SystemMessage(content="""
You are a concise assistant.
Always respond:
- In 5-6 bullet points
- Short and simple
- No tables
- No markdown formatting
"""),
        HumanMessage(content="What is the economy of Bangladesh?")
    ]
}
chat_bot.invoke(initial_state)['messages'][-1].content

'- Rapid growth, GDP ~ $400\u202fbillion, 6‑7% annual expansion.  \n- Service sector dominates (≈60% of GDP), followed by industry and agriculture.  \n- Major exports: ready‑made garments, textiles, leather, jute, seafood.  \n- Large remittance inflow from overseas workers, about 10% of GDP.  \n- Challenges: infrastructure gaps, energy shortages, high population density, climate vulnerability.  \n- Government focuses on industrialization, digital economy, and climate‑resilient development.'

In [38]:
thread_id ='1'
while True:
    user_message = input('type here:')
    print('user:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id':thread_id}}
    response = chat_bot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI:', response['messages'][-1].content)


user: hi
AI: Hello! How can I help you today?
user: my name is mohammad
AI: Nice to meet you, Mohammad! How can I assist you today?
user: can you tell me my name?
AI: Sure! Your name is Mohammad.
user: can you tell me mohammad meaning in bangla
AI: **মহম্মদ (Mohammad)**  
বাংলা ভাষায় এই নামের অর্থ হলো **“প্রশংসিত”** বা **“যাকে প্রশংসা করা হয়”**।  

- নামটি আরবি মূল “ḥ‑m‑d” থেকে এসেছে, যার অর্থ “প্রশংসা করা”।  
- ইসলামের পবিত্র গ্রন্থে, **মহম্মদ** (মুহাম্মদ) হলেন নবী মুহাম্মদের নাম, যাকে মুসলিমরা “প্রশংসিত” ও “সর্বোত্তম” বলে মনে করেন।  
- বাংলাদেশে ও পাকিস্তানে এই নামটি খুবই প্রচলিত, এবং সাধারণত “মহম্মদ” বা “মোহাম্মদ” বানান ব্যবহার করা হয়।  

সুতরাং, বাংলায় “মহম্মদ” বলতে বোঝায় **একজন যাকে সর্বদা প্রশংসা করা হয়**।
user: exit
